## Inference of TinyLlama LoRA model

In [1]:
import os
import math

import torch
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, default_data_collator
from peft import PeftModel

## Load in model

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "./tinyllama_lora_ft"

# Quantization config
bits_and_bytes_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load in model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bits_and_bytes_config,
    device_map="auto",
    trust_remote_code=True,
).eval()

# Load in tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Load in model temp model
tmp_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bits_and_bytes_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load in tuned model and merge 
tuned_model = PeftModel.from_pretrained(tmp_model, adapter_path)
tuned_model = tuned_model.merge_and_unload().eval()


print(tuned_model)

/home/arre/anaconda3/envs/llm/lib/python3.12/site-packages/peft/tuners/lora/bnb.py:351: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), e

## Tokenization function

In [4]:
def tokenize(batch):
    texts = [
        f"### Instruction:\n{instruction}\n### Reponse:\n{output}"
        for instruction, output in zip(batch["question"], batch["answer"])
    ]

    tokens = tokenizer(
        texts,
        padding = "max_length",
        truncation = True,
        max_length = 256,
        return_tensors = "pt"
    )

    tokens["labels"] = tokens["input_ids"].clone()

    return tokens

## Tokenize dataset

In [5]:
eval_data = load_dataset("openai/gsm8k", "main", split="train[:200]")
eval_data = eval_data.map(tokenize, batched=True, remove_columns=["question", "answer"])
eval_data = eval_data.with_format("torch")


In [6]:
eval_loader = DataLoader(
    eval_data,
    batch_size=8,
    collate_fn=default_data_collator
)

In [7]:
@torch.no_grad()
def compute_perplexity(model, dataloader):
    model.eval()
    loss = 0
    for batch in eval_loader:
        batch = {k: v.to("cuda") for k,v in batch.items()}
        outputs = model(**batch)
        loss += outputs.loss.item()
    perplexity = math.exp(loss / len(dataloader))
    return perplexity

In [9]:
print(f"Base model perplexity: {compute_perplexity(base_model, eval_loader)}")
print(f"Tuned model perplexity: {compute_perplexity(tuned_model, eval_loader)}")

Base model perplexity: 196.103280608608
Tuned model perplexity: 41.04641512689968
